In [ ]:
# 1. Install Document Extraction & Parsing (Lightning-fast Markdown approach)
!pip install pymupdf4llm langchain-text-splitters pypdf==4.2.0

# 2. Install Vector Database
!pip install chromadb==0.4.18

# 3. Install Embeddings & Deep Learning Foundations
!pip install torch>=2.0.0 sentence-transformers==3.0.1 hf_xet

# 4. Install AI Orchestration Frameworks & Gateways
!pip install langchain==0.2.5 langchain-community==0.2.5 openai==1.35.3 litellm==1.41.1

# 5. Install Evaluation, Observability & SRE Metrics
!pip install deepeval==4.1.1 langsmith

In [ ]:
!pip install onnxruntime-gpu

In [2]:
# Force install onnxruntime (CPU version) to avoid CUDA dependency issues, overriding any previously installed GPU version
#!pip install --force-reinstall onnxruntime

import pymupdf4llm
import re
import json
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Define your Colab Drive paths
pdf_path = "/content/drive/MyDrive/ET_Industrial_data/regulatory/factories_act_1948.pdf"
cache_path = "/content/drive/MyDrive/ET_Industrial_data/regulatory/factories_act_chunks_cache.json"

def markdown_regex_parse_factories_act(pdf_path):
    print("⏳ Extracting document natively to Markdown...")
    md_text = pymupdf4llm.to_markdown(pdf_path)

    print("⏳ Chunking via Markdown bold tags...")
    section_pattern = re.compile(r'(\n\*\*\d+[A-Z]?\.\s+[^\*]+\*\*)')
    parts = section_pattern.split(md_text)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=150)

    chunks = []
    chunk_idx = 0

    if parts[0].strip():
        preamble_splits = text_splitter.split_text(parts[0].strip())
        for split_txt in preamble_splits:
            chunks.append({
                "text": split_txt,
                "metadata": {"source": "factories_act_1948.pdf", "page": 1, "section_title": "Preamble / TOC", "chunk_id": f"chunk_{chunk_idx}"}
            })
            chunk_idx += 1

    for i in range(1, len(parts), 2):
        header_raw = parts[i].strip()
        body_raw = parts[i+1].strip()

        clean_title = header_raw.replace('**', '').split('.—')[0].split('—')[0].strip()
        combined_text = f"{header_raw}\n{body_raw}"

        section_splits = text_splitter.split_text(combined_text)

        for split_txt in section_splits:
            chunks.append({
                "text": split_txt,
                "metadata": {
                    "source": "factories_act_1948.pdf",
                    "page": 1,
                    "section_title": clean_title,
                    "chunk_id": f"chunk_{chunk_idx}"
                }
            })
            chunk_idx += 1

    return chunks

# Check if we already processed this to save time
if os.path.exists(cache_path):
    print("🚀 Bypassing parser. Loading pre-computed chunks from Drive...")
    with open(cache_path, 'r', encoding='utf-8') as f:
        docs = json.load(f)
else:
    docs = markdown_regex_parse_factories_act(pdf_path)
    print("⏳ Saving parsed chunks to Google Drive...")
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(docs, f, ensure_ascii=False, indent=4)

print(f"🔥 Total Semantic Chunks Ready: {len(docs)}\n")
for d in docs[15:20]:
    print(f"🔹 [{d['metadata']['chunk_id']}] | Section: {d['metadata']['section_title']}")
    print(f"Text snippet: {d['text'][:280]}...\n" + "-"*50)

🚀 Bypassing parser. Loading pre-computed chunks from Drive...
🔥 Total Semantic Chunks Ready: 212

🔹 [chunk_15] | Section: 5. Power to exempt during public emergency
Text snippet: **5. Power to exempt during public emergency.—**
In any case of public emergency the State Government may, by notification in the Official Gazette, exempt any factory or class or description of factories from all or any of the provisions of this Act<sup>4</sup> [except section 67...
--------------------------------------------------
🔹 [chunk_16] | Section: 6. Approval, licensing and registration of factories
Text snippet: **6. Approval, licensing and registration of factories.—**
( _1_ ) The State Government may make rules **—** 

6 [( _a_ ) requiring, for the purposes of this Act, the submission of plans of any class or description of factories to the Chief Inspector or the State Government;] 

7...
--------------------------------------------------
🔹 [chunk_17] | Section: 6. Approval, licensing and registrat

In [3]:
import chromadb
from chromadb.utils import embedding_functions

# Initialize local PyTorch-backed HuggingFace embedding engine
local_emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

# Create persistent local storage client
chroma_client = chromadb.PersistentClient(path="./data/vectorstore/")
collection = chroma_client.get_or_create_collection(name="industrial_brain_v1", embedding_function=local_emb_fn)

# Clear old vector runs if re-running the cell
if collection.count() > 0:
    all_ids = collection.get()["ids"]
    collection.delete(ids=all_ids)

# Load chunks into Chroma
collection.add(
    documents=[d["text"] for d in docs],
    metadatas=[d["metadata"] for d in docs],
    ids=[d["metadata"]["chunk_id"] for d in docs]
)

print(f"💾 Local ChromaDB successfully loaded with {collection.count()} persistent records.")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


💾 Local ChromaDB successfully loaded with 212 persistent records.


In [ ]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

In [12]:
from litellm import completion
from google.colab import userdata

def expand_query_for_legal_rag(original_query):
    # Quick, low-latency LLM call to generate search keywords
    prompt = (
        f"You are a legal search optimizer. Expand the following user query into a list of "
        f"exact legal terms, synonyms, and section title concepts likely found in the 1948 Factories Act. "
        f"Output ONLY the keywords separated by spaces. Do not write full sentences.\n"
        f"Query: {original_query}"
    )

    response = completion(
        model="gemini/gemini-3.5-flash",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,
        api_key=userdata.get('GEMINI_API_KEY')
    )
    expanded_terms = response.choices[0].message.content.strip()
    # Combine original query with the expanded legal terms
    return f"{original_query} {expanded_terms}"

def run_optimized_retrieval(query_str):
    # 1. Expand the query behind the scenes
    search_vector_query = expand_query_for_legal_rag(query_str)
    print(f"🔮 Original: '{query_str}'\n🚀 Expanded Search Terms: '{search_vector_query}'")

    # 2. Query ChromaDB with the expanded term set
    results = collection.query(query_texts=[search_vector_query], n_results=5)

    print(f"\n==================================================")
    print(f"🔍 SEARCH QUERY: '{query_str}'")
    print(f"==================================================")

    for idx in range(len(results['documents'][0])):
        chunk_id = results['ids'][0][idx]
        score = results['distances'][0][idx]
        meta = results['metadatas'][0][idx]
        text_content = results['documents'][0][idx]

        print(f"📍 Match {idx+1} | ID: {chunk_id} | Distance Score: {score:.4f}")
        print(f"   Source: {meta['source']} | Section: {meta['section_title']} | Page: {meta['page']}")
        print(f"   Text: {text_content[:200]}...\n")

# Re-run your test
run_optimized_retrieval("confined space entry requirements")

🔮 Original: 'confined space entry requirements'
🚀 Expanded Search Terms: 'confined space entry requirements Section 36 Section 36A confined space dangerous fumes gas vapour dust chamber tank vat pit pipe flue manhole breathing apparatus reviving apparatus safety belt rope ventilation exhaust portable electric light written permission certificate entry precautions safety measures hazard risk permit-to-work asphyxiation toxic atmosphere oxygen deficiency entry requirements safety equipment protective gear legal compliance Indian Factories Act 1948 Section 36 Section 36A Section 37'

🔍 SEARCH QUERY: 'confined space entry requirements'
📍 Match 1 | ID: chunk_66 | Distance Score: 0.6787
   Source: factories_act_1948.pdf | Section: 35. Protection of eyes | Page: 1
   Text: **35. Protection of eyes.—**
In respect of any such manufacturing process carried on in any factory as may be prescribed, being a process which involves **—** 

( _a_ ) risk of injury to the eyes from...

📍 Match 2 | ID: ch

In [13]:
from litellm import completion
from google.colab import userdata

def test_optimized_generation(query_str):
    # 1. Expand the query behind the scenes for better search
    expanded_query = expand_query_for_legal_rag(query_str)

    # 2. Fetch Top 3 chunks using the EXPANDED query
    retrieved = collection.query(query_texts=[expanded_query], n_results=3)

    context_blocks = []
    for i in range(len(retrieved['documents'][0])):
        c_id = retrieved['ids'][0][i]
        sec = retrieved['metadatas'][0][i]['section_title']
        p_num = retrieved['metadatas'][0][i]['page']
        txt = retrieved['documents'][0][i]
        context_blocks.append(f"--- ID: {c_id} (Section: {sec}, Page: {p_num})---\n{txt}")

    full_context_str = "\n\n".join(context_blocks)

    system_prompt = (
        "You are an industrial compliance inspector. Synthesize an answer for the query using ONLY the provided text blocks. "
        "Every single factual claim you make must end with a precise inline bracket citation matching the source ID exactly (e.g. [chunk_12]). "
        "If the provided context chunks do not contain enough specific facts to fully answer a question, explicitly state: "
        "'INFO NOT IN CONTEXT'. Do not assume, complete, or pull external engineering rules."
    )

    user_prompt = f"CONTEXT CHUNKS:\n{full_context_str}\n\nQUERY: {query_str}"

    # 3. Execute LLM generation
    response = completion(
        model="gemini/gemini-3.5-flash",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.0,
        api_key=userdata.get('GEMINI_API_KEY')
    )

    answer = response.choices[0].message.content

    print(f"\n==================================================")
    print(f"🤖 LLM RESPONSE GENERATION FOR: '{query_str}'")
    print(f"==================================================")
    print(f"✍️ Generated Output:\n{answer}\n")

# Run the final test!
test_optimized_generation("confined space entry requirements")


🤖 LLM RESPONSE GENERATION FOR: 'confined space entry requirements'
✍️ Generated Output:
Based on the provided compliance standards, the requirements for entering a confined space (such as a chamber, tank, vat, pit, pipe, flue, or other confined space) in a factory are as follows:

*   **Egress and Access:** No person shall be required or allowed to enter the confined space unless it is provided with a manhole of adequate size or another effective means of egress [chunk_66].
*   **Atmospheric Hazard Mitigation:** Before entry, all practicable measures must be taken to remove any gas, fume, vapour, or dust to bring its level within permissible limits and to prevent any ingress of such substances [chunk_66].
*   **Entry Authorization and Safety Gear:** Entry is prohibited unless at least one of the following conditions is met:
    *   A competent person has issued a written certificate, based on a test they carried out personally, confirming that the space is reasonably free from dangero

In [14]:
test_optimized_generation("What safety measures apply to fencing of machinery?")
test_optimized_generation("What are the provisions for hazardous processes?")


🤖 LLM RESPONSE GENERATION FOR: 'What safety measures apply to fencing of machinery?'
✍️ Generated Output:
Based on the provided text, the safety measures for the fencing of machinery in every factory require that specific machinery parts must be securely fenced by safeguards of substantial construction [chunk_46]. 

These safeguards must be constantly maintained and kept in position while the parts of machinery they are fencing are in motion or in use [chunk_46]. 

The specific parts that require this fencing include:
*   Every moving part of a prime mover and every flywheel connected to a prime mover, regardless of whether the prime mover or flywheel is in the engine house or not [chunk_46].
*   The headrace and tailrace of every water-wheel and water turbine [chunk_46].
*   Any part of a stock-bar that projects beyond the head stock of a lathe [chunk_46].
*   The following parts, unless they are in such a position or of such construction as to be as safe to every person employed in 